In [18]:
%%time
%matplotlib inline

import importlib
import new_import_ODC  

importlib.reload(new_import_ODC)

from new_import_ODC import *

CPU times: user 1.15 s, sys: 0 ns, total: 1.15 s
Wall time: 1.13 s


In [2]:
%%time
import os
import sys

print("✅ AWS credentials loaded from environment variables")

# Force reload datacube module để lấy cấu hình mới
if 'datacube' in sys.modules:
    del sys.modules['datacube']
if 'datacube.config' in sys.modules:
    del sys.modules['datacube.config']

# Cấu hình Dask local (không dùng EasiDefaults)
from dask.distributed import Client, LocalCluster

cluster = LocalCluster(n_workers=4)
client = Client(cluster)

# Khai báo Datacube cục bộ
# Datacube sẽ đọc cấu hình từ ~/.datacube.conf
import datacube
try:
    dc = datacube.Datacube()
    print("✅ Datacube connected successfully to local database!")
    print(f"   Database: {dc.index.url}")
except Exception as e:
    print(f"❌ Error connecting to Datacube: {e}")
    print("Please check PostgreSQL is running and ~/.datacube.conf is configured correctly")
    dc = None

# Cấu hình truy cập dịch vụ S3
configure_s3_access(aws_unsigned=False, requester_pays=True, client=client)
print("✅ S3 access configured")

client

✅ AWS credentials loaded from environment variables
✅ Datacube connected successfully to local database!
   Database: postgresql://x79@/var/run/postgresql:5432/datacube
✅ Datacube connected successfully to local database!
   Database: postgresql://x79@/var/run/postgresql:5432/datacube
✅ S3 access configured
CPU times: user 768 ms, sys: 153 ms, total: 921 ms
Wall time: 2.92 s
✅ S3 access configured
CPU times: user 768 ms, sys: 153 ms, total: 921 ms
Wall time: 2.92 s


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 24,Total memory: 31.26 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:37681,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:44469,Total threads: 6
Dashboard: http://127.0.0.1:36577/status,Memory: 7.82 GiB
Nanny: tcp://127.0.0.1:43113,


In [3]:
%%time
# Tự động add product vào datacube nếu chưa có
import subprocess
import yaml

print("🔄 Checking and adding Sentinel-2 product to datacube...")

# Đọc product definition
product_yaml = "s2_l2a_product.yaml"
try:
    # Thử add product (sẽ skip nếu đã tồn tại)
    result = subprocess.run(
        ["datacube", "product", "add", product_yaml],
        capture_output=True,
        text=True,
        check=False
    )
    
    if "Added" in result.stdout or "Updated" in result.stdout:
        print(f"✅ Product added/updated from {product_yaml}")
    elif "already exists" in result.stdout.lower() or "already exists" in result.stderr.lower():
        print(f"ℹ️  Product 's2_l2a' already exists in datacube")
    else:
        print(f"📋 Product status: {result.stdout}")
    
    # Liệt kê products
    result = subprocess.run(
        ["datacube", "product", "list"],
        capture_output=True,
        text=True
    )
    print("\n📦 Available products in datacube:")
    print(result.stdout)
    
except Exception as e:
    print(f"⚠️  Warning: Could not add product: {e}")
    print("Continuing anyway...")

print("\n" + "="*60)

🔄 Checking and adding Sentinel-2 product to datacube...
📋 Product status: 
📋 Product status: 

📦 Available products in datacube:
s2_l2a  Sentinel-2 Level 2A Surface Reflectance


CPU times: user 465 ms, sys: 74.2 ms, total: 539 ms
Wall time: 5.07 s

📦 Available products in datacube:
s2_l2a  Sentinel-2 Level 2A Surface Reflectance


CPU times: user 465 ms, sys: 74.2 ms, total: 539 ms
Wall time: 5.07 s


In [4]:
%%time
# Load Sentinel-2 metadata from datacube & display RGB
print("="*80)
print("SENTINEL-2 METADATA LOADER (Direct S3 COG Access - No Datacube API)")
print("="*80)

import numpy as np
import matplotlib.pyplot as plt

# Get datasets from datacube
print("\n[1] Loading metadata from datacube...")
datasets = list(dc.find_datasets(product='s2_l2a', time=("2023-03-01", "2023-12-31")))
print(f"OK: Loaded {len(datasets)} scenes from datacube")

# Filter by spatial bbox
bbox_target = {'min_x': 105.5, 'max_x': 106.4, 'min_y': 9.2, 'max_y': 10.0}
filtered_scenes = []

for ds in datasets:
    extent = ds.extent
    geom = extent.geom if hasattr(extent, 'geom') else extent
    bounds = geom.bounds
    ds_bbox = {'min_x': bounds[0], 'min_y': bounds[1], 'max_x': bounds[2], 'max_y': bounds[3]}
    
    if not (bbox_target['max_x'] < ds_bbox['min_x'] or 
            bbox_target['min_x'] > ds_bbox['max_x'] or
            bbox_target['max_y'] < ds_bbox['min_y'] or
            bbox_target['min_y'] > ds_bbox['max_y']):
        filtered_scenes.append(ds)

print(f"OK: Filtered to {len(filtered_scenes)} scenes in study area")

# Display available scenes
print("\n" + "="*80)
print("[2] AVAILABLE SCENES")
print("="*80)
for i, ds in enumerate(filtered_scenes[:20]):
    scene_time = ds.time.begin if hasattr(ds.time, 'begin') else ds.time
    print(f"  [{i:2d}] {ds.metadata.label:50s} {scene_time.strftime('%Y-%m-%d')}")
if len(filtered_scenes) > 20:
    print(f"  ... and {len(filtered_scenes) - 20} more scenes")

print(f"\nOK: Stored {len(filtered_scenes)} scenes in memory for RGB loading")
print("="*80)


SENTINEL-2 METADATA LOADER (Direct S3 COG Access - No Datacube API)

[1] Loading metadata from datacube...
OK: Loaded 40 scenes from datacube
OK: Filtered to 40 scenes in study area

[2] AVAILABLE SCENES
  [ 0] S2A_48PWR_20231226_0_L2A                           2023-12-26
  [ 1] S2A_48PWS_20231226_0_L2A                           2023-12-26
  [ 2] S2A_48PXS_20231226_0_L2A                           2023-12-26
  [ 3] S2B_48PWR_20231221_0_L2A                           2023-12-21
  [ 4] S2B_48PXR_20231221_0_L2A                           2023-12-21
  [ 5] S2B_48PWS_20231221_0_L2A                           2023-12-21
  [ 6] S2B_48PXS_20231221_0_L2A                           2023-12-21
  [ 7] S2A_48PWR_20231216_0_L2A                           2023-12-16
  [ 8] S2A_48PXR_20231216_0_L2A                           2023-12-16
  [ 9] S2A_48PWS_20231216_0_L2A                           2023-12-16
  [10] S2A_48PXS_20231216_0_L2A                           2023-12-16
  [11] S2B_48PXR_20231211_0_L2A      

In [5]:
%%time
# 🔧 Check if data files are actually accessible
print("🔍 Checking data file accessibility...")

try:
    # Get first matching scene
    datasets = list(dc.find_datasets(
        product='s2_l2a',
        time=("2023-03-01", "2023-12-31")
    ))
    
    if datasets:
        selected = datasets[0]
        print(f"\n📦 Dataset: {selected.metadata.label}")
        print(f"   ID: {selected.id}")
        print(f"   Metadata type: {type(selected.metadata)}")
        
        # Check if measurements are in dataset
        print(f"\n📊 Measurements:")
        if hasattr(selected, 'measurements'):
            for name, measurement in selected.measurements.items():
                print(f"   - {name}: {measurement}")
        
        # Try to access the file paths
        print(f"\n📂 URI info:")
        if hasattr(selected, 'uris'):
            for uri in selected.uris:
                print(f"   {uri}")
        
        # Try to check file existence
        print(f"\n✔️ Checking if data is from S3 or local...")
        if hasattr(selected, 'uris'):
            for uri in selected.uris:
                if 's3://' in uri:
                    print(f"   🌐 S3 URI detected: {uri[:60]}...")
                elif 'file://' in uri or '/' in uri:
                    print(f"   💾 Local file: {uri}")
        
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*60)


🔍 Checking data file accessibility...

📦 Dataset: S2A_48PWR_20231226_0_L2A
   ID: 94d5e7dd-4877-5f4d-9b30-e671f38f59a0
   Metadata type: <class 'datacube.utils.documents.DocReader'>

📊 Measurements:
   - nir: {'band': 1, 'path': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/48/P/WR/2023/12/S2A_48PWR_20231226_0_L2A/B08.tif'}
   - red: {'band': 1, 'path': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/48/P/WR/2023/12/S2A_48PWR_20231226_0_L2A/B04.tif'}
   - scl: {'band': 1, 'path': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/48/P/WR/2023/12/S2A_48PWR_20231226_0_L2A/SCL.tif'}
   - blue: {'band': 1, 'path': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/48/P/WR/2023/12/S2A_48PWR_20231226_0_L2A/B02.tif'}
   - green: {'band': 1, 'path': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/48/P/WR/2023/12/S2A_48PWR_20231226_0_L2A/B03.tif'}
   - nir08: {'band': 1, 'path': 'https

In [6]:
%%time
# 💾 LOAD DATA DIRECTLY FROM S3 COGS USING RASTERIO
print("🚀 Loading Sentinel-2 data directly from S3 COGs...")

try:
    import rasterio
    from rasterio.io import MemoryFile
    import xarray as xr
    import numpy as np
    
    # Get first matching scene
    datasets = list(dc.find_datasets(
        product='s2_l2a',
        time=("2023-03-01", "2023-12-31")
    ))
    
    if datasets:
        selected = datasets[0]
        print(f"\n📦 Scene: {selected.metadata.label}")
        scene_datetime = selected.time.begin if hasattr(selected.time, 'begin') else selected.time
        
        # Get measurements
        measurements_to_load = ['red', 'green', 'blue', 'nir', 'scl']
        data_dict = {}
        
        print(f"\n⏳ Loading bands from S3 COGs...")
        for band_name in measurements_to_load:
            if band_name in selected.measurements:
                band_path = selected.measurements[band_name]['path']
                print(f"   📥 {band_name}: {band_path[:70]}...")
                
                try:
                    with rasterio.open(band_path) as src:
                        # Read data with proper bounds clipping
                        # Target bbox in lat/lon: (105.5, 9.2, 106.4, 10.0)
                        data = src.read(1)
                        data_dict[band_name] = data
                        print(f"      ✅ Loaded shape: {data.shape}, dtype: {data.dtype}")
                except Exception as e:
                    print(f"      ⚠️ Could not load {band_name}: {e}")
        
        if data_dict:
            print(f"\n✅ Successfully loaded {len(data_dict)} bands!")
            print(f"\n📊 Data shapes:")
            for name, data in data_dict.items():
                print(f"   {name}: {data.shape}, min={data.min()}, max={data.max()}")
        else:
            print("❌ Could not load any bands")
    
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*60)


🚀 Loading Sentinel-2 data directly from S3 COGs...

📦 Scene: S2A_48PWR_20231226_0_L2A

⏳ Loading bands from S3 COGs...
   📥 red: https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/...


      ✅ Loaded shape: (10980, 10980), dtype: uint16
   📥 green: https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/...
      ✅ Loaded shape: (10980, 10980), dtype: uint16
   📥 blue: https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/...
      ✅ Loaded shape: (10980, 10980), dtype: uint16
   📥 blue: https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/...
      ✅ Loaded shape: (10980, 10980), dtype: uint16
   📥 nir: https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/...
      ✅ Loaded shape: (10980, 10980), dtype: uint16
   📥 nir: https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/...
      ✅ Loaded shape: (10980, 10980), dtype: uint16
   📥 scl: https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/...
      ✅ Loaded shape: (10980, 10980), dtype: uint16
   📥 scl: https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/...
      ✅ Loaded shape: (5490, 5490), dtype: u

In [12]:
## ⚙️ Configuration: Set date range and spatial extent
print("="*70)
print("CONFIGURATION: Date Range & Spatial Extent")
print("="*70)

# Date range for Sentinel-2 data query
# Available data: 2023-03-08 to 2023-12-26
date_range = ("2023-03-01", "2023-12-31")
print(f"\n📅 Date range: {date_range[0]} to {date_range[1]}")

# Longitude range (min, max) in EPSG:4326
# Study area: Chau Thanh district, Vietnam
longtitude_range = (105.5, 106.4)
print(f"🧭 Longitude range: {longtitude_range[0]} to {longtitude_range[1]}")

# Latitude range (min, max) in EPSG:4326
latitude_range = (9.2, 10.0)
print(f"🧭 Latitude range: {latitude_range[0]} to {latitude_range[1]}")

# Coordinates for Sentinel-1 SAR data (if needed)
coordinates = {
    'x': longtitude_range,
    'y': latitude_range
}
print(f"\n✅ All configuration variables set and ready to use")
print("="*70)

CONFIGURATION: Date Range & Spatial Extent

📅 Date range: 2023-03-01 to 2023-12-31
🧭 Longitude range: 105.5 to 106.4
🧭 Latitude range: 9.2 to 10.0

✅ All configuration variables set and ready to use


In [ ]:
## 📊 Load Sentinel-2 Data (materialized on Dask workers)
print("\n" + "="*70)
print("LOADING SENTINEL-2 DATA")
print("="*70)

data = load_data(dc, date_range, longtitude_range, latitude_range)
print(f"\n✅ Data loaded successfully!")
print(f"\n📊 Dataset info:")
notebook_utils.heading(notebook_utils.xarray_object_size(data))
display(data)

print("="*70)


LOADING SENTINEL-2 DATA
Loading Sentinel-2 data (EPSG:32648)...
  Time range: ('2023-03-01', '2023-12-31')
  Measurements: ['red', 'nir', 'scl']
✅ Data loaded successfully!
   Dimensions: {'time': 20, 'y': 2, 'x': 1}
   Time steps: 20
   Spatial extent: x=1, y=2

⏳ Loading data into memory...


2025-11-02 23:24:42,637 - distributed.worker - ERROR - Compute Failed
Key:       ('dc_load_scl-3caf3455909e4882acb3956bd46162e9', 15, 0, 0)
State:     executing
Task:  <Task ('dc_load_scl-3caf3455909e4882acb3956bd46162e9', 15, 0, 0) fuse_lazy(...)>
Exception: 'AttributeError("\'dict\' object has no attribute \'nodata\'")'
Traceback: '  File "/home/x79/miniconda/envs/env_01/lib/python3.10/site-packages/datacube/api/core.py", line 930, in fuse_lazy\n    data = numpy.full(geobox.shape, measurement.nodata, dtype=measurement.dtype)\n'

2025-11-02 23:24:42,638 - distributed.worker - ERROR - Compute Failed
Key:       ('dc_load_red-870233f913f74fcc90187c580199d486', 15, 0, 0)
State:     executing
Task:  <Task ('dc_load_red-870233f913f74fcc90187c580199d486', 15, 0, 0) fuse_lazy(...)>
Exception: 'AttributeError("\'dict\' object has no attribute \'nodata\'")'
Traceback: '  File "/home/x79/miniconda/envs/env_01/lib/python3.10/site-packages/datacube/api/core.py", line 930, in fuse_lazy\n    data = 

AttributeError: 'dict' object has no attribute 'nodata'

In [ ]:
data_computed = data.compute()
print(notebook_utils.xarray_object_size(data_computed))  # Will show actual MB

NameError: name 'data' is not defined

In [ ]:
## Diagnostic: Check actual data size (Dask vs computed)
print("="*70)
print("DATA SIZE ANALYSIS")
print("="*70)

print(f"\n1️⃣ LAZY-LOADED (Dask) size: {notebook_utils.xarray_object_size(data)}")
print("   (Shows 0.00 MB because data is not yet materialized)")

print(f"\n2️⃣ ACTUAL DATA STRUCTURE:")
print(f"   Variables: {list(data.data_vars)}")
print(f"   Dimensions: {dict(data.dims)}")
print(f"   Coordinates: {list(data.coords)}")

# Show actual size if we were to compute it
print(f"\n3️⃣ ESTIMATED SIZE IF COMPUTED:")
for var in data.data_vars:
    nbytes = np.prod(data[var].shape) * 8 / (1024**2)  # Assume 8 bytes per value
    print(f"   {var}: {data[var].shape} → ~{nbytes:.1f} MB")

# Check Dask chunks
print(f"\n4️⃣ DASK CHUNK INFO:")
for var in data.data_vars:
    print(f"   {var}: chunks = {data[var].chunks}")

print("="*70)

DATA SIZE ANALYSIS

1️⃣ LAZY-LOADED (Dask) size: Dataset size: 0.00 MB
   (Shows 0.00 MB because data is not yet materialized)

2️⃣ ACTUAL DATA STRUCTURE:
   Variables: ['red', 'nir', 'scl']
   Dimensions: {'time': 20, 'y': 2, 'x': 1}
   Coordinates: ['time', 'y', 'x', 'spatial_ref']

3️⃣ ESTIMATED SIZE IF COMPUTED:
   red: (20, 2, 1) → ~0.0 MB
   nir: (20, 2, 1) → ~0.0 MB
   scl: (20, 2, 1) → ~0.0 MB

4️⃣ DASK CHUNK INFO:
   red: chunks = ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (2,), (1,))
   nir: chunks = ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (2,), (1,))
   scl: chunks = ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (2,), (1,))


/tmp/ipykernel_6669/2592752281.py:11: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"   Dimensions: {dict(data.dims)}")


In [ ]:
# Test cell: Kiểm tra dữ liệu có sẵn trong datacube
print("🔍 Testing datacube query...")
print(f"Date range in config: {date_range}")
print(f"Longitude range: {longtitude_range}")
print(f"Latitude range: {latitude_range}")

# ⚠️ Test với date range đúng (dữ liệu có từ 2023-03-08 đến 2023-12-26)
test_date_range = ("2023-03-01", "2023-12-31")
print(f"\n🧪 Testing with corrected date range: {test_date_range}")

try:
    # Test 1: Query datasets available (dùng geopolygon thay vì lon/lat)
    print("\n1️⃣ Checking available datasets...")
    from datacube.utils.geometry import box
    
    # Tạo bounding box
    bbox = box(longtitude_range[0], latitude_range[0], 
               longtitude_range[1], latitude_range[1], crs='EPSG:4326')
    
    datasets = dc.find_datasets(
        product='s2_l2a',
        time=test_date_range,
        geopolygon=bbox  # Sử dụng geopolygon thay vì lon/lat
    )
    dataset_list = list(datasets)
    print(f"   Found {len(dataset_list)} datasets matching query")
    
    if dataset_list:
        print(f"\n📦 First dataset: {dataset_list[0].metadata.label}")
        print(f"   Time: {dataset_list[0].time}")
        print(f"\n📦 Last dataset: {dataset_list[-1].metadata.label}")
        print(f"   Time: {dataset_list[-1].time}")
        
        # Hiển thị chi tiết một scene bất kỳ (chọn scene ở giữa)
        mid_idx = len(dataset_list) // 2
        sample_dataset = dataset_list[mid_idx]
        print(f"\n🎯 Sample dataset (scene #{mid_idx + 1} of {len(dataset_list)}):")
        print(f"   Label: {sample_dataset.metadata.label}")
        print(f"   Time: {sample_dataset.time}")
        print(f"   ID: {sample_dataset.id}")
        print(f"   CRS: {sample_dataset.crs}")
        print(f"   Extent: {sample_dataset.extent}")
        
        # Hiển thị các measurements có sẵn
        print(f"\n📊 Available measurements:")
        for name, measurement in sample_dataset.measurements.items():
            print(f"   - {name}: {measurement['path']}")
    
        # Test 2: Load một scene để hiển thị
        print("\n2️⃣ Loading a single scene for visualization...")
        # Lấy thời gian của scene ở giữa
        mid_scene = dataset_list[mid_idx]
        scene_time = mid_scene.time
        
        # Load scene đó với window nhỏ
        single_scene = dc.load(
            product='s2_l2a',
            x=longtitude_range,
            y=latitude_range,
            time=(scene_time, scene_time),
            measurements=['red', 'green', 'blue', 'nir', 'scl'],
            output_crs='EPSG:4326',
            resolution=(-0.0001, 0.0001),
            group_by='solar_day',
        )
        
        if single_scene.dims and len(single_scene.data_vars) > 0:
            print(f"   ✅ Scene loaded successfully!")
            print(f"   Variables: {list(single_scene.data_vars)}")
            print(f"   Dimensions: {dict(single_scene.dims)}")
            print(f"   Size: {notebook_utils.xarray_object_size(single_scene)}")
            display(single_scene)
            
            # Hiển thị ảnh RGB nếu có đủ bands
            if all(b in single_scene.data_vars for b in ['red', 'green', 'blue']):
                print("\n🖼️ Displaying RGB preview...")
                import matplotlib.pyplot as plt
                
                # Tạo RGB composite (scale đơn giản)
                rgb = single_scene[['red', 'green', 'blue']].to_array().squeeze()
                rgb = rgb / rgb.quantile(0.98)  # Normalize
                rgb = rgb.clip(0, 1)
                
                plt.figure(figsize=(10, 10))
                plt.imshow(rgb.transpose('y', 'x', 'variable'))
                plt.title(f"RGB Preview: {mid_scene.metadata.label}")
                plt.axis('off')
                plt.show()
        else:
            print(f"   ⚠️ No data returned for single scene")
    else:
        print(f"   ⚠️ No datasets found. Check date range and spatial extent.")
        
except Exception as e:
    print(f"   ❌ Error: {e}")
    import traceback
    traceback.print_exc()

🔍 Testing datacube query...
Date range in config: ('2023-03-01', '2023-12-31')
Longitude range: (104.5, 107.0)
Latitude range: (8.5, 11.0)

🧪 Testing with corrected date range: ('2023-03-01', '2023-12-31')

1️⃣ Checking available datasets...
   Found 0 datasets matching query
   ⚠️ No datasets found. Check date range and spatial extent.


In [ ]:
# Test query với range rộng hơn để xem có data không
print("🔍 Testing with expanded range...")
test_range = (9.8, 10.9)  # Phủ toàn bộ dữ liệu có sẵn
test_data = load_data(dc, date_range, longtitude_range, test_range)
print(f"Test dataset size: {notebook_utils.xarray_object_size(test_data)}")
print(f"Shape: {test_data.dims}")
display(test_data)

🔍 Testing with expanded range...
Loading Sentinel-2 data (EPSG:32648)...
  Time range: ('2023-03-01', '2023-12-31')
  Measurements: ['red', 'nir', 'scl']
✅ Data loaded successfully!
   Dimensions: {'time': 20, 'y': 2, 'x': 1}
   Time steps: 20
   Spatial extent: x=1, y=2
Test dataset size: Dataset size: 0.00 MB
Shape: FrozenMappingWarningOnValuesAccess({'time': 20, 'y': 2, 'x': 1})


<xarray.Dataset> Size: 388B
Dimensions:      (time: 20, y: 2, x: 1)
Coordinates:
  * time         (time) datetime64[ns] 160B 2023-03-08T03:25:12.032000 ... 20...
  * y            (y) float64 16B 15.0 5.0
  * x            (x) float64 8B 105.0
    spatial_ref  int32 4B 32648
Data variables:
    red          (time, y, x) uint16 80B dask.array<chunksize=(1, 2, 1), meta=np.ndarray>
    nir          (time, y, x) uint16 80B dask.array<chunksize=(1, 2, 1), meta=np.ndarray>
    scl          (time, y, x) uint8 40B dask.array<chunksize=(1, 2, 1), meta=np.ndarray>
Attributes:
    crs:           epsg:32648
    grid_mapping:  spatial_ref

In [ ]:
%%time
# Tiến hành loại bỏ các vị trí bị mây ảnh hưởng
result = mask_clean(data)
progress(result)

✅ Cloud masking applied
   Good pixel classes: [4, 5, 6, 7]
   Mask created (dask-backed, not yet computed)
   Data variables masked: ['red', 'nir']
   Result persisted to workers
CPU times: user 79.9 ms, sys: 25.4 ms, total: 105 ms
Wall time: 97.3 ms


VBox()

2025-11-02 21:41:44,315 - distributed.worker - ERROR - Compute Failed
Key:       ('dc_load_red-af7c0994ffa34996a0d723036a7cc41d', 12, 0, 0)
State:     executing
Task:  <Task ('dc_load_red-af7c0994ffa34996a0d723036a7cc41d', 12, 0, 0) fuse_lazy(...)>
Exception: 'AttributeError("\'dict\' object has no attribute \'nodata\'")'
Traceback: '  File "/home/x79/miniconda/envs/env_01/lib/python3.10/site-packages/datacube/api/core.py", line 930, in fuse_lazy\n    data = numpy.full(geobox.shape, measurement.nodata, dtype=measurement.dtype)\n'

2025-11-02 21:41:44,316 - distributed.worker - ERROR - Compute Failed
Key:       ('dc_load_nir-25ee9d5ad2bb4552bd4d706d2487750d', 12, 0, 0)
State:     executing
Task:  <Task ('dc_load_nir-25ee9d5ad2bb4552bd4d706d2487750d', 12, 0, 0) fuse_lazy(...)>
Exception: 'AttributeError("\'dict\' object has no attribute \'nodata\'")'
Traceback: '  File "/home/x79/miniconda/envs/env_01/lib/python3.10/site-packages/datacube/api/core.py", line 930, in fuse_lazy\n    data = 

In [ ]:
# Tiến hành tính toán NDVI
ds1 = calculate_indices(result, index="NDVI", satellite_mission="s2")
ndvi = ds1["NDVI"]
display(ndvi)

NameError: name 'result' is not defined

In [ ]:
## Hiển thị ảnh NDVI chưa điền các giá trị mây (chưa fill nan)
plt.imshow(ndvi.isel(time=6))

In [ ]:
# Thiết lập giá trị trung bình mùa vụ để xử lý các điểm ảnh bị mây dựa vào sự thay đổi theo mùa
time_split = [
    slice("2022-09-01", "2023-01-01"),
    slice("2023-01-01", "2023-05-01"),
    slice("2023-05-01", "2023-07-01"),
    slice("2023-07-01", "2023-10-01"),
]

# Điền mây ở các vị trí mang giá trị nan (fill nan)
fill_nan_ndvi = fill_nan(ndvi, time_split)

# In kết quả ảnh NDVI đã điền mây (đã fill nan)
plt.imshow(fill_nan_ndvi.isel(time=6))

In [ ]:
%%time
## tính ndvi theo tháng
average_ndvi = fill_nan_ndvi.resample(time="1M").mean().persist()
progress(average_ndvi)

# compute average_ndvi
average_ndvi = average_ndvi.compute()

In [ ]:
#Load dữ liệu ảnh Sentinel 1
dsvh, dsvv = load_data_sen1(dc, date_range, coordinates)
average_vv = calculate_average(dsvv, time_pattern='1M')
average_vh = calculate_average(dsvh, time_pattern='1M')

In [ ]:
## cấu hình bộ dữ liệu điểm huấn luyện mô hình (train file)
train_path = "train/ST_training data_updated_1130points_new.shp"  # đường dẫn shp file train

## load dữ liệu điểm huấn luyện mô hình (train file)
train = load_train_data(train_path)
train.head()

# cấu hình nhãn dữ liệu 
label_mapping = {
    "Lua tom": "0",
    "Lua": "1",
    "CHN": "2",
    "CLN": "3",
    "TS": "4",
    "Song": "5",
    "Dat xay dung": "6",
    "Rung": "7",
}

# xây dựng tập dữ liệu (dataset) chứa dữ liệu VH, VV, NDVI
datasets = get_data_sen1_and_sen2(train, average_ndvi, average_vh, average_vv)

# chia tập dữ liệu thành các phần theo tỉ lệ 80(80-20)-20 tương ứng với tập train, validate, test
X_train, X_val, X_test, y_train, y_val, y_test = split_train_data(
    train, label_mapping, datasets
)

In [ ]:
# Huấn luyện mô hình
grid_search = train_with_rf(X_train, X_val, y_train, y_val)

In [ ]:
# kiểm tra độ chính xác với tập test
y_pred_test = grid_search.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred_test)
print(f"Accuracy for test data {round(test_accuracy, 2)*100} %")

In [ ]:
# Lưu mô hình huấn luyện
save_model("model_odc.joblib", grid_search)

In [ ]:
# đóng client, cluster
client.close()
cluster.close()